# Chapter 1 &mdash; Hilbert's Program: Undecidable vs. Incomplete

**Concept 3 of the Chapter 1 decomposition:** *Hilbert's Program, and its Refutation: Undecidability and Incompleteness*

Hilbert wanted one algorithm to decide every mathematical statement. G&ouml;del, Church and Turing showed it cannot exist &mdash; in two <i>different</i> ways that are easy to confuse.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1-Intro/Concept-Hilbert-Undecidable-Incomplete/Concept-Hilbert-Undecidable-Incomplete.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Hilbert demanded an **algorithm** &mdash; systematic, mechanical, and *terminating on any
input* &mdash; to decide the truth of **any** mathematical statement.

Two distinct failures killed it:

* **Undecidable**: no algorithm decides truth of statements in the system.
* **Incomplete**: some *true* statements cannot be *proved* inside the system.

The first is about **algorithms**. The second is about **proofs**. Students merge
them constantly; this notebook keeps them apart.

## 2. Definitions

### A decider, a semi-decider, and the difference

We model a tiny "system" of statements about naturals, and three attitudes a machine
can take toward them.

In [ ]:
# A statement is a predicate on naturals plus a claim "this holds for ALL n".
forall_even_plus_odd_is_odd = lambda n: (2*n + 1) % 2 == 1     # true for all n
forall_n_squared_lt_n       = lambda n: n*n < n                # false (n=0,1,2..)

def semi_decide_forall(pred, budget=1000):
    """Halts with 'FALSE' if it finds a counterexample.
       If the claim is true, it searches forever -- we cut it off."""
    for n in range(budget):
        if not pred(n):
            return ('FALSE', 'counterexample n=%d' % n)
    return ('UNKNOWN', 'no counterexample within %d' % budget)

### Why the asymmetry is the whole story

Refuting "for all $n$, P(n)" needs **one** witness &mdash; a finite object you can exhibit.
Confirming it needs **all** of them &mdash; an infinite job. So "false" is discoverable
and "true" is not. That is *semi-decidability*, and it is what Hilbert's program ran into.

<!-- nav-strip -->

---

&larr;&nbsp;[Ch1&nbsp;2.&nbsp;Problem vs. Procedure vs. Algorithm](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1-Intro/Concept-Problem-Procedure-Algorithm/Concept-Problem-Procedure-Algorithm.ipynb) &nbsp;&middot;&nbsp; [**Chapter 1** index](https://github.com/ganeshutah/Jove/blob/master/Chapter1-Intro/README.md) &nbsp;&middot;&nbsp; [Ch1&nbsp;4.&nbsp;Hilbert's Tenth Problem, Diophantine Equations, and the MRDP Theorem](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1-Intro/Concept-Diophantine-MRDP/Concept-Diophantine-MRDP.ipynb)&nbsp;&rarr;

---

## 3. Tests

The false claim is refuted quickly and the machine **halts**.

In [ ]:
print("n^2 < n for all n?  ->", semi_decide_forall(forall_n_squared_lt_n))
assert semi_decide_forall(forall_n_squared_lt_n)[0] == 'FALSE'

The true claim is never confirmed &mdash; the machine simply runs out of budget.
`UNKNOWN` is the honest answer, and no budget upgrades it to `TRUE`.

In [ ]:
print("2n+1 odd for all n? ->", semi_decide_forall(forall_even_plus_odd_is_odd))
print("... with 100x budget ->", semi_decide_forall(forall_even_plus_odd_is_odd, 100000))
assert semi_decide_forall(forall_even_plus_odd_is_odd)[0] == 'UNKNOWN'
print("No budget ever upgrades UNKNOWN to TRUE.")

**Undecidable vs incomplete, side by side.** Both are failures, but of different things.

In [ ]:
print("UNDECIDABLE : no algorithm decides truth      (about ALGORITHMS)")
print("INCOMPLETE  : some truths have no proof       (about PROOFS)")
print()
print("A system can be undecidable and complete, or decidable and incomplete.")
print("They are independent properties -- do not merge them.")

## 4. Exercises


1. Modify `semi_decide_forall` so it also halts with `TRUE` when the predicate is
   true for all $n$. You cannot &mdash; say precisely where you get stuck.
2. Give a claim about naturals that is *true* but for which the smallest
   counterexample search would run past $10^{12}$. (Hint: look up P&oacute;lya's conjecture.)
   What does this do to your confidence in "no counterexample found"?
3. In one sentence each, state Hilbert's demand, G&ouml;del's answer, and Turing's answer.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter1-Intro/Concept-Hilbert-Undecidable-Incomplete')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')